# Site figures (redrawn for the website, Nov 6)

Draws every figure panel as its own small SVG, in a light and a dark version, into `site/img/` (new files).
The titles and captions are on the web page, below each panel, so the images carry none.
The original figures in `img/` are not touched.

Inputs: `data/ladder_2026.csv`, `data/ablation_groups.csv`, `data/ablation_blocks.csv`, `data/metric_check_2026.csv` (repo);
`bundle_2026.parquet` and `site_figdata_seasons.csv` (Drive, read-only).

In [1]:
# ===== F0. Setup and style (COPY-PASTE) =====
import os, json
import numpy as np
import pandas as pd
import matplotlib as mpl
mpl.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

if os.path.exists('/content'):
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/IPL PROJECT ALL DOCS/data/'
    REPO = '/content/ml_death_overs_ipl/'
else:
    DRIVE = os.environ.get('SITE_DRIVE', 'drive/')
    REPO = os.environ.get('SITE_REPO', '../')
OUT = REPO + 'site/img/'
os.makedirs(OUT, exist_ok=True)

THEME = {   # the same colours as the Phase 2 figures and the site
    'light': dict(surface='#fcfcfb', ink='#0b0b0b', ink2='#52514e', grid='#e6e5e0', zero='#8a8984', data='#3b3a37',
                  past='#8f8e88', band='#f0efec', model={'xgb': '#2a78d6', 'lr': '#eb6834', 'b1': '#1baf7a'}),
    'dark':  dict(surface='#1a1a19', ink='#f0efec', ink2='#c3c2b7', grid='#33332f', zero='#8a8984', data='#d6d5cf',
                  past='#77766f', band='#262624', model={'xgb': '#3987e5', 'lr': '#d95926', 'b1': '#199e70'}),
}
MARKER = {'xgb': 'o', 'lr': 's', 'b1': '^'}
NAMES = ['0', '1', '2', '3', '4', '6', 'W']

def use_theme(mode):
    T = THEME[mode]
    mpl.rcParams.update({
        'figure.facecolor': T['surface'], 'axes.facecolor': T['surface'], 'savefig.facecolor': T['surface'],
        'text.color': T['ink'], 'axes.labelcolor': T['ink2'], 'xtick.color': T['ink2'], 'ytick.color': T['ink2'],
        'axes.edgecolor': T['grid'], 'axes.grid': True, 'grid.color': T['grid'], 'grid.linewidth': 0.8,
        'axes.axisbelow': True, 'axes.spines.top': False, 'axes.spines.right': False,
        'font.size': 10, 'axes.labelsize': 10, 'legend.frameon': False, 'lines.linewidth': 1.6,
        'svg.fonttype': 'none', 'font.family': 'sans-serif', 'svg.hashsalt': 'site'})
    return T

saved = []
def render(draw, name, size):
    """draw(ax, T) draws one panel; saved once per theme as site/img/<name>_<theme>.svg"""
    for mode in ('light', 'dark'):
        T = use_theme(mode)
        fig, ax = plt.subplots(figsize=size, layout='constrained')
        draw(ax, T)
        fig.savefig(OUT + f'{name}_{mode}.svg', bbox_inches='tight', pad_inches=0.06, metadata={'Date': None})
        plt.close(fig)
    saved.append(name)

def interval_marks(ax, yv, mid, lo, hi, color, filled, surface, marker='o'):
    for yy, m, l, h, f in zip(yv, mid, lo, hi, filled):
        ax.plot([l, h], [yy, yy], color=color, lw=1.6, solid_capstyle='round')
        ax.plot(m, yy, marker=marker, ms=7, color=color, mfc=color if f else surface, mew=1.6)
print('ready')

ready


In [2]:
# ===== F1. Drift by season: one panel per outcome (COPY-PASTE) =====
SEAS = DRIVE + 'site_figdata_seasons.csv'
C = pd.read_csv(SEAS)
by_year = C.pivot_table(index='year', columns='y', values='balls', aggfunc='sum').fillna(0)
rate = by_year.div(by_year.sum(axis=1), axis=0) * 100
tr = C[C['split'] == 'train'].groupby('y')['balls'].sum()
print('1. legal balls:', int(C['balls'].sum()))                                        # expect 276360
print(f"2. six %: 2008-2024 {100 * tr[5] / tr.sum():.1f} | 2025 {rate.loc[2025, 5]:.1f} | 2026 {rate.loc[2026, 5]:.1f}")  # 5.1 | 7.9 | 8.5
for k, key in [(5, 'sixes'), (4, 'fours'), (0, 'dots'), (6, 'wickets')]:
    def draw(ax, T, k=k):
        s = rate[k]
        ax.axvspan(2024.5, 2026.5, color=T['band'], zorder=0, lw=0)
        ax.plot(s.index, s.values, color=T['data'], marker='o', ms=3.5)
        ax.annotate(f'{s.iloc[-1]:.1f}%', (s.index[-1], s.iloc[-1]), xytext=(5, 0), textcoords='offset points',
                    va='center', color=T['ink'], fontsize=9)
        ax.set_xlim(2007, 2029); ax.set_xticks([2008, 2014, 2020, 2026])
        ax.yaxis.set_major_locator(MaxNLocator(5))
        ax.set_ylabel('% of legal balls')
    render(draw, f'drift_{key}', (3.3, 2.5))
print('3. drawn: drift panels')

1. legal balls: 276360
2. six %: 2008-2024 5.1 | 2025 7.9 | 2026 8.5


3. drawn: drift panels


In [3]:
# ===== F2. Outcome shares by phase: one panel per phase (COPY-PASTE) =====
PH = ['powerplay', 'middle', 'death']
def shares(rows):
    t = rows.pivot_table(index='match_phase', columns='y', values='balls', aggfunc='sum').reindex(index=PH, columns=range(7)).fillna(0)
    return t.div(t.sum(axis=1), axis=0) * 100
def entropy(p): p = p[p > 0] / 100; return float(-(p * np.log(p)).sum())
PAST, NOW = shares(C[C['year'] <= 2025]), shares(C[C['year'] == 2026])
ENT = {ph: (round(entropy(PAST.loc[ph].values), 2), round(entropy(NOW.loc[ph].values), 2)) for ph in PH}
print('1. entropy 2008-2025 vs 2026:', ENT)        # Phase 2: pp 1.43/1.52, middle 1.41/1.46, death 1.60/1.60
top = 1.08 * max(PAST.values.max(), NOW.values.max())
for ph in PH:
    def draw(ax, T, ph=ph):
        x, w = np.arange(7), 0.4
        ax.bar(x - w / 2, PAST.loc[ph].values, width=w - 0.04, color=T['past'], label='2008-2025')
        ax.bar(x + w / 2, NOW.loc[ph].values, width=w - 0.04, color=T['data'], label='2026')
        ax.set_xticks(x, NAMES); ax.grid(axis='x', visible=False); ax.set_ylim(0, top)
        ax.set_xlabel('outcome (runs, or W = wicket)'); ax.set_ylabel('% of legal balls')
    render(draw, f'phase_{ph}', (3.4, 2.6))
print('2. drawn: phase panels')

1. entropy 2008-2025 vs 2026: {'powerplay': (1.43, 1.52), 'middle': (1.41, 1.46), 'death': (1.6, 1.6)}


2. drawn: phase panels


In [4]:
# ===== F3. Where the gain over B1 comes from (COPY-PASTE) =====
lad = pd.read_csv(REPO + 'data/ladder_2026.csv').query("metric == 'logloss'")
STEPS = ['scorecard state', '+ player type', '+ in-match context', '+ history (X-hist)']
LAB = ['finer scorecard', 'batter and\nbowler type', 'in-match context', 'history of\nearlier matches']
for balls in ['death', 'all']:
    def draw(ax, T, balls=balls):
        r = lad[lad['balls'] == balls].set_index('rung').loc[STEPS]
        yv = np.arange(4)[::-1]
        interval_marks(ax, yv, -r['step'], -r['step_high'], -r['step_low'], T['model']['xgb'],
                       (r['step_high'] < 0).values, T['surface'])
        for yy, red, sh in zip(yv, -r['step'], r['share_of_gain_over_B1']):
            ax.annotate(f'{100 * sh:.0f}%', (red, yy), xytext=(0, 8), textcoords='offset points', ha='center', fontsize=9, color=T['ink'])
        ax.axvline(0, color=T['zero'], lw=1)
        ax.set_yticks(yv, LAB); ax.set_ylim(-0.5, 3.6)
        ax.xaxis.set_major_locator(MaxNLocator(4))
        ax.set_xlabel('drop in log-loss per ball (higher = helps more)')
    render(draw, f'ladder_{balls}', (4.6, 2.9))
print('drawn: ladder')

drawn: ladder


In [5]:
# ===== F4. Dropping one group at a time (COPY-PASTE) =====
ab = pd.read_csv(REPO + 'data/ablation_groups.csv').query('season == 2026').rename(columns={'group': 'what'})
bl = pd.read_csv(REPO + 'data/ablation_blocks.csv').query('season == 2026').rename(columns={'block': 'what'})
abl = pd.concat([ab, bl])
ROWS = [('delivery', 'delivery'), ('chase', 'chase pressure'), ('partnership', 'partnership'),
        ('striker', 'striker form'), ('bowler_spell', 'bowler spell'), ('momentum', 'momentum'),
        ('hist_8_batter', 'batter history'), ('hist_9_bowler', 'bowler history'),
        ('hist_10_base', 'matchup and base rates'), ('hist_11_venue_season', 'venue and season'),
        ('all_inmatch_context', 'ALL in-match context'), ('all_history', 'ALL history')]
for balls in ['death', 'all']:
    def draw(ax, T, balls=balls):
        yv = np.array([13, 12, 11, 10, 9, 8, 6, 5, 4, 3, 1, 0])
        r = abl[abl['balls'] == balls].set_index('what').loc[[k for k, _ in ROWS]]
        interval_marks(ax, yv, r['loss_if_dropped'], r['low'], r['high'], T['model']['xgb'], (r['low'] > 0).values, T['surface'])
        ax.axvline(0, color=T['zero'], lw=1)
        ax.set_yticks(yv, [lab for _, lab in ROWS])
        ax.xaxis.set_major_locator(MaxNLocator(4))
        ax.set_xlabel('rise in log-loss when removed')
    render(draw, f'ablation_{balls}', (4.6, 4.0))
print('drawn: ablation')

drawn: ablation


In [6]:
# ===== F5. Calibration at the death: dots, sixes, wickets (COPY-PASTE) =====
Bn = pd.read_parquet(DRIVE + 'bundle_2026.parquet')
D = Bn[Bn['death'].astype(bool)]
yd = D['y'].to_numpy().astype(int)
P = {m: D[[f'p{t}_{k}' for k in range(7)]].to_numpy() for m, t in [('xgb', 'x'), ('lr', 'l'), ('b1', 'b1')]}
print('1. death balls:', len(yd))                                               # expect 3583
def rel_points(p, hit, n_bins=10):
    b = pd.qcut(p, n_bins, labels=False, duplicates='drop')
    g = pd.DataFrame({'b': b, 'p': p, 'hit': hit}).groupby('b')
    return 100 * g['p'].mean().values, 100 * g['hit'].mean().values
for k, key in [(0, 'dots'), (5, 'sixes'), (6, 'wickets')]:
    hit = (yd == k).astype(float)
    pts = {m: rel_points(P[m][:, k], hit) for m in P}
    def draw(ax, T, pts=pts):
        top = 1.08 * max(max(a.max(), b.max()) for a, b in pts.values())
        low = 0.85 * min(min(a.min(), b.min()) for a, b in pts.values())
        ax.plot([low, top], [low, top], color=T['zero'], lw=1, ls='--')
        for m, (px, ob) in pts.items():
            ax.plot(px, ob, color=T['model'][m], marker=MARKER[m], ms=5, lw=1.5)
        ax.set_xlim(low, top); ax.set_ylim(low, top); ax.set_aspect('equal')
        ax.set_xlabel('predicted %'); ax.set_ylabel('actual %')
    render(draw, f'calib_{key}', (3.2, 3.1))
print('2. drawn: calibration')

1. death balls: 3583


2. drawn: calibration


In [7]:
# ===== F6. Same verdict under log-loss and Brier (COPY-PASTE) =====
mc = pd.read_csv(REPO + 'data/metric_check_2026.csv')
for c in ('low', 'high'):
    mc[c + '_pct'] = 100 * mc[c] / mc['B']
PAIRS = [('X-hist - B1', 'XGBoost vs B1'), ('X-hist - LR (B2-hist +V)', 'XGBoost vs\nlogistic regression'),
         ('LR (B2-hist +V) - B1', 'logistic regression\nvs B1'), ('B1 - B0', 'B1 vs B0')]
for balls in ['death', 'all']:
    def draw(ax, T, balls=balls):
        yv = np.arange(4)[::-1]
        for metric, dy, mk, col in [('logloss', 0.14, 'o', T['data']), ('brier', -0.14, 's', T['ink2'])]:
            r = mc[(mc['balls'] == balls) & (mc['metric'] == metric)].set_index('pair').loc[[p for p, _ in PAIRS]]
            for yy, m, l, h in zip(yv + dy, r['pct'], r['low_pct'], r['high_pct']):
                ax.plot([l, h], [yy, yy], color=col, lw=1.6, solid_capstyle='round')
                ax.plot(m, yy, marker=mk, ms=7, color=col, mew=1.6, mfc=col if metric == 'logloss' else T['surface'])
        ax.axvline(0, color=T['zero'], lw=1)
        ax.set_yticks(yv, [lab for _, lab in PAIRS])
        ax.set_xlabel('% change in loss (left of 0 = first model better)')
    render(draw, f'headline_{balls}', (4.6, 2.9))
print('drawn: headline |', len(saved), 'panels saved:', saved)

drawn: headline | 16 panels saved: ['drift_sixes', 'drift_fours', 'drift_dots', 'drift_wickets', 'phase_powerplay', 'phase_middle', 'phase_death', 'ladder_death', 'ladder_all', 'ablation_death', 'ablation_all', 'calib_dots', 'calib_sixes', 'calib_wickets', 'headline_death', 'headline_all']
